# R3Con — provenance

How to check an answer: the memos and the sentences it rests on, the summaries and records it used, and the run folder that keeps them. The memos are the [quickstart](quickstart.ipynb)'s.

In [1]:
import json
import os
import textwrap

from r3con import r3con

# The key comes from the provider's own variable, or a .env beside this notebook.
assert os.environ.get("OPENAI_API_KEY") or os.path.exists(".env"), (
    "Set OPENAI_API_KEY first."
)

MODEL = "openai/gpt-6-luna"  # any litellm model string

QUESTION = (
    "Which maintenance contractor's sites logged the most equipment incidents in Q3 "
    "in total, and how many?"
)

## The answer

`r3con.run` reads the five memos and answers; printing the result prints the answer.

In [2]:
docs = r3con.read_documents("memos")
result = r3con.run(question=QUESTION, documents=docs, model=MODEL)
print(result)

Halloran Services Ltd’s sites logged the most equipment incidents in Q3, with 11 total (5 at Northgate and 6 at Riverside).


## The memos behind it

`result.provenance["documents"]` lists the memos behind the summaries the answer cites, directly or through the records it cites, each by its id and its file name.

In [3]:
for document in result.provenance["documents"]:
    print(document["document"], document["name"])

4cfc71 01_northgate.txt
8cd551 02_eastfield.txt
59cca8 03_riverside.txt
882b42 04_contractor_registry.txt


## The sentences behind it

Each quote is a sentence of a memo the answer rests on. `document_text` is the memo's own words there, from `start` to `end`, shown here between » and « with a few words of the memo on either side; `text` is the sentence as the model quoted it, shown too where it differs; and `match` says how it was found, with a `score` for a `fuzzy` one.

In [4]:
CHARS_AROUND = 40
WORDS_AROUND = 3

for quote in result.provenance["quotes"]:
    text, start, end = docs[quote["name"]], quote["start"], quote["end"]
    before = text[max(0, start - CHARS_AROUND) : start].split()[-WORDS_AROUND:]
    after = text[end : end + CHARS_AROUND].split()[:WORDS_AROUND]
    score = "" if quote["score"] is None else f" {quote['score']}"
    print(f"{quote['name']} ({quote['match']}{score})")
    print("  " + " ".join([*before, f"»{quote['document_text']}«", *after]))
    if quote["text"] != quote["document_text"]:
        print(f"  quoted as: {quote['text']}")

01_northgate.txt (exact)
  Q3 operations review »Equipment incidents logged this quarter: 5.« Three were conveyor
01_northgate.txt (exact)
  were HVAC trips. »Planned maintenance at this site is carried out under contractor code CT-118.« Site manager: Dana
02_eastfield.txt (exact)
  Q3 operations review »Equipment incidents logged this quarter: 9.« Mostly compressor faults,
02_eastfield.txt (exact)
  clustered in August. »Planned maintenance at this site is carried out under contractor code CT-204.« Site manager: Sam
03_riverside.txt (exact)
  »MEMO — Riverside site, Q3 operations review« Equipment incidents logged
03_riverside.txt (exact)
  Q3 operations review »Equipment incidents logged this quarter: 6.« Four were electrical,
03_riverside.txt (exact)
  storm in September. »Planned maintenance at this site is carried out under contractor code CT-118.« Site manager: Priya
04_contractor_registry.txt (exact)
  (extract, revised July) »CT-118 .... Halloran Services Ltd — mechanical and e

## Which summaries and records it used

R3Con writes a summary of each memo for the question, kept in the run folder's `relevance/result.json`, and parses each memo into records; the answer cites the summaries and the memos' records it used by id, and `name_of` turns an id into its file name.

In [5]:
names = dict(zip(result.document_ids, result.document_names, strict=True))


def name_of(id_: str) -> str:
    """The file name of a memo by its id, or by <id>.<k> for its part k."""
    return names[id_.partition(".")[0]]


relevance = json.loads((result.run_dir / "relevance" / "result.json").read_text())
final_round = relevance["rounds"][-1]
final = {summary["id"]: summary for summary in final_round["snippets"]}

print("Summaries it cites:")
for id_ in result.provenance["summary_ids"]:
    print(" ", name_of(id_))
    print(textwrap.indent(textwrap.fill(final[id_]["summary"]), "    "))

print("\nMemos whose records it used:")
for id_ in result.provenance["parse_ids"]:
    print(" ", name_of(id_))

Summaries it cites:
  01_northgate.txt
    A Q3 operations memo for the Northgate site reports 5 equipment
    incidents, adding 5 incidents to contractor code CT-118’s total. The
    contractor registry identifies CT-118 as Halloran Services Ltd; the
    Riverside memo also reports incidents under that code.
  03_riverside.txt
    A Q3 operations-review memo for the Riverside site. It reports 6
    equipment incidents under maintenance contractor code CT-118, which
    the contractor registry identifies as Halloran Services Ltd. This adds
    6 incidents to that contractor’s total; Northgate also used CT-118.
  02_eastfield.txt
    A Q3 operations-review memo for Eastfield. It reports 9 equipment
    incidents under contractor code CT-204, which the contractor registry
    identifies as Merrow Facilities Group.
  04_contractor_registry.txt
    An approved contractor registry maps site-paperwork codes to
    contractor names. It identifies CT-118 as Halloran Services Ltd and
    CT-204

Each summary was written with the other memos' summaries from the round before in view, and its `provenance` says which ones it drew on.

In [6]:
print("the answer")
for id_ in result.provenance["summary_ids"]:
    print(f"└─ {name_of(id_)} (round {final_round['round']})")
    for earlier in final[id_]["provenance"]:
        print(f"   └─ {name_of(earlier)} (round {final_round['round'] - 1})")

the answer
└─ 01_northgate.txt (round 2)
   └─ 03_riverside.txt (round 1)
   └─ 04_contractor_registry.txt (round 1)
└─ 03_riverside.txt (round 2)
   └─ 01_northgate.txt (round 1)
   └─ 04_contractor_registry.txt (round 1)
└─ 02_eastfield.txt (round 2)
   └─ 04_contractor_registry.txt (round 1)
└─ 04_contractor_registry.txt (round 2)
   └─ 01_northgate.txt (round 1)
   └─ 02_eastfield.txt (round 1)
   └─ 03_riverside.txt (round 1)


## Records

The records parsed from the memos whose records it used, from `result.structured_context`, with each record's memo id from `result.source_ids`.

In [7]:
for field, records in result.structured_context.items():
    print(field)
    for record, document in zip(records, result.source_ids[field], strict=True):
        if document in result.provenance["parse_ids"]:
            print(f"  {name_of(document)}: {record}")

equipment_incident_reports
  01_northgate.txt: {'document': '4cfc71', 'site': 'Northgate', 'period': 'Q3', 'incident_type': 'equipment incidents', 'incident_count': 5, 'contractor_code': 'CT-118', 'contractor_role': 'planned maintenance'}
  02_eastfield.txt: {'document': '8cd551', 'site': 'Eastfield', 'period': 'Q3', 'incident_type': 'equipment incidents', 'incident_count': 9, 'contractor_code': 'CT-204', 'contractor_role': 'planned maintenance'}
  03_riverside.txt: {'document': '59cca8', 'site': 'Riverside', 'period': 'Q3', 'incident_type': 'equipment incidents', 'incident_count': 6, 'contractor_code': 'CT-118', 'contractor_role': 'planned maintenance'}
contractor_registry_entries
  04_contractor_registry.txt: {'document': '882b42', 'contractor_code': 'CT-118', 'contractor_name': 'Halloran Services Ltd'}
  04_contractor_registry.txt: {'document': '882b42', 'contractor_code': 'CT-204', 'contractor_name': 'Merrow Facilities Group'}
  04_contractor_registry.txt: {'document': '882b42', 'c

## In the run folder

`result.run_dir` keeps all of this as files:

- `ids.json` maps each id to its memo's place in `docs`, and `names.json` to its file name.
- `relevance/result.json` holds every round's summaries, each with its quotes, the sentences of its memo it is based on (`text`, the model's copy, `start` and `end`, `document_text`, the memo's own words there, and how it was found, `match` and `score`), and the summaries of the round before it drew on (`provenance`).
- `structuring/parsing/result.json` holds the records, and the summaries each memo's parse drew on.
- `reasoning/result.json` holds the answer, the summaries and records it cites, the memos behind them (`documents`), and its quotes.